In [14]:
# ========== I1：IPTW 稳定化权重 ==========
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression

df_iptw = df_main.copy()
df_iptw['age_75_84']  = (df_iptw['age_group'] == '75-84').astype(int)
df_iptw['age_85plus'] = (df_iptw['age_group'] == '85+').astype(int)

ps_vars = ['gender_male', 'age_75_84', 'age_85plus',
           'baseline_k', 'drug_count_no_laxatives',
           'chf', 'ckd', 'dm',
           'diuretic', 'acei_arb', 'digoxin', 'insulin',
           'statin', 'anticoagulant']

df_iptw = df_iptw.dropna(subset=ps_vars + ['senna_exposed', 'y']).copy()

# 倾向评分
X = df_iptw[ps_vars].values
y_ps = df_iptw['senna_exposed'].values
lr = LogisticRegression(max_iter=2000, solver='lbfgs')
lr.fit(X, y_ps)
df_iptw['ps'] = lr.predict_proba(X)[:, 1]

# 稳定化权重
p_treat = df_iptw['senna_exposed'].mean()
df_iptw['w'] = np.where(
    df_iptw['senna_exposed'] == 1,
    p_treat / df_iptw['ps'],
    (1 - p_treat) / (1 - df_iptw['ps'])
)

print('原始权重分布:')
print(df_iptw['w'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]))

# 截尾（1%–99% 分位）—— 防止极端权重
w_lo = df_iptw['w'].quantile(0.01)
w_hi = df_iptw['w'].quantile(0.99)
df_iptw['w_trim'] = df_iptw['w'].clip(w_lo, w_hi)

print(f'\n截尾阈值: [{w_lo:.3f}, {w_hi:.3f}]')
print('截尾后权重分布:')
print(df_iptw['w_trim'].describe())

原始权重分布:
count    41529.000000
mean         0.999735
std          0.174481
min          0.574370
1%           0.690539
5%           0.769055
25%          0.882795
50%          0.974840
75%          1.088297
95%          1.310566
99%          1.547497
max          4.353826
Name: w, dtype: float64

截尾阈值: [0.691, 1.547]
截尾后权重分布:
count    41529.000000
mean         0.998179
std          0.165219
min          0.690539
25%          0.882795
50%          0.974840
75%          1.088297
max          1.547497
Name: w_trim, dtype: float64


In [15]:
# ========== I2：IPTW 加权后 SMD ==========
import numpy as np
import pandas as pd

def smd_binary_w(x, g, w):
    x = x.values.astype(float); g = g.values; w = w.values
    p1 = np.average(x[g==1], weights=w[g==1])
    p0 = np.average(x[g==0], weights=w[g==0])
    pooled = np.sqrt((p1*(1-p1) + p0*(1-p0)) / 2)
    return abs(p1 - p0) / pooled if pooled > 0 else np.nan

def smd_continuous_w(x, g, w):
    x = x.values.astype(float); g = g.values; w = w.values
    m1 = np.average(x[g==1], weights=w[g==1])
    m0 = np.average(x[g==0], weights=w[g==0])
    v1 = np.average((x[g==1] - m1)**2, weights=w[g==1])
    v0 = np.average((x[g==0] - m0)**2, weights=w[g==0])
    pooled = np.sqrt((v1 + v0) / 2)
    return abs(m1 - m0) / pooled if pooled > 0 else np.nan

def smd_binary(x, g):
    p0, p1 = x[g==0].mean(), x[g==1].mean()
    pooled = np.sqrt((p0*(1-p0) + p1*(1-p1)) / 2)
    return abs(p1 - p0) / pooled if pooled > 0 else np.nan

def smd_continuous(x, g):
    x0, x1 = x[g==0].dropna(), x[g==1].dropna()
    pooled = np.sqrt((x0.std()**2 + x1.std()**2) / 2)
    return abs(x1.mean() - x0.mean()) / pooled if pooled > 0 else np.nan

binary_set = {'gender_male', 'age_75_84', 'age_85plus', 'chf', 'ckd', 'dm',
              'diuretic', 'acei_arb', 'digoxin', 'insulin', 'statin',
              'anticoagulant'}

g_col = df_iptw['senna_exposed']
w_col = df_iptw['w_trim']

rows = []
for v in ps_vars:
    if v in binary_set:
        s_raw  = smd_binary(df_iptw[v].astype(int), g_col)
        s_iptw = smd_binary_w(df_iptw[v].astype(int), g_col, w_col)
    else:
        s_raw  = smd_continuous(df_iptw[v], g_col)
        s_iptw = smd_continuous_w(df_iptw[v], g_col, w_col)
    rows.append({'Variable': v,
                 'SMD before': round(s_raw, 3),
                 'SMD after IPTW': round(s_iptw, 3),
                 'Balanced': '✓' if s_iptw < 0.1 else '✗'})

smd_iptw = pd.DataFrame(rows)
print(smd_iptw.to_string(index=False))
print(f'\nIPTW 后 SMD<0.1 的变量: '
      f'{(smd_iptw["SMD after IPTW"]<0.1).sum()}/{len(smd_iptw)}')
print(f'IPTW 后 SMD 最大: {smd_iptw["SMD after IPTW"].max():.3f}')
smd_iptw.to_excel('TableS_IPTW_SMD.xlsx', index=False)

               Variable  SMD before  SMD after IPTW Balanced
            gender_male       0.047           0.000        ✓
              age_75_84       0.006           0.001        ✓
             age_85plus       0.102           0.006        ✓
             baseline_k       0.174           0.012        ✓
drug_count_no_laxatives       0.203           0.014        ✓
                    chf       0.083           0.004        ✓
                    ckd       0.039           0.002        ✓
                     dm       0.017           0.000        ✓
               diuretic       0.023           0.002        ✓
               acei_arb       0.013           0.001        ✓
                digoxin       0.000           0.000        ✓
                insulin       0.009           0.001        ✓
                 statin       0.046           0.001        ✓
          anticoagulant       0.156           0.005        ✓

IPTW 后 SMD<0.1 的变量: 14/14
IPTW 后 SMD 最大: 0.014


In [16]:
# ========== I3：IPTW 加权回归 ==========
import statsmodels.api as sm
import statsmodels.formula.api as smf

formula_iptw = (
    'y ~ senna_exposed + gender_male + '
    'C(age_group, Treatment("65-74")) + baseline_k + '
    'drug_count_no_laxatives + '
    'chf + ckd + dm + diuretic + acei_arb + digoxin + '
    'insulin + statin + anticoagulant'
)

model_iptw = smf.glm(formula_iptw, data=df_iptw,
                     family=sm.families.Binomial(),
                     freq_weights=df_iptw['w_trim'].values).fit()

or_ = np.exp(model_iptw.params['senna_exposed'])
lo  = np.exp(model_iptw.conf_int().loc['senna_exposed', 0])
hi  = np.exp(model_iptw.conf_int().loc['senna_exposed', 1])
p   = model_iptw.pvalues['senna_exposed']
print(f'IPTW 加权 senna aOR = {or_:.3f} ({lo:.3f}–{hi:.3f}), P = {p:.3g}')

IPTW 加权 senna aOR = 1.198 (1.138–1.262), P = 6.16e-12


In [17]:
# ========== I4：重叠权重 ==========
df_iptw['w_ow'] = np.where(
    df_iptw['senna_exposed'] == 1,
    df_iptw['ps'],
    1 - df_iptw['ps']
)

print('重叠权重分布:')
print(df_iptw['w_ow'].describe())

model_ow = smf.glm(formula_iptw, data=df_iptw,
                   family=sm.families.Binomial(),
                   freq_weights=df_iptw['w_ow'].values).fit()

or_ = np.exp(model_ow.params['senna_exposed'])
lo  = np.exp(model_ow.conf_int().loc['senna_exposed', 0])
hi  = np.exp(model_ow.conf_int().loc['senna_exposed', 1])
p   = model_ow.pvalues['senna_exposed']
print(f'\n重叠权重 senna aOR = {or_:.3f} ({lo:.3f}–{hi:.3f}), P = {p:.3g}')

重叠权重分布:
count    41529.000000
mean         0.527401
std          0.113978
min          0.095697
25%          0.441748
50%          0.531267
75%          0.612390
max          0.928308
Name: w_ow, dtype: float64

重叠权重 senna aOR = 1.168 (1.082–1.262), P = 7.76e-05


In [18]:
# ========== I5：IPTW 汇总 ==========
summary_iptw = pd.DataFrame([
    {'Method': 'IPTW (stabilized, trimmed)',
     'senna aOR': round(np.exp(model_iptw.params['senna_exposed']), 3),
     '95% CI': f"{np.exp(model_iptw.conf_int().loc['senna_exposed',0]):.3f}–"
               f"{np.exp(model_iptw.conf_int().loc['senna_exposed',1]):.3f}",
     'P': f"{model_iptw.pvalues['senna_exposed']:.3g}"},
    {'Method': 'Overlap weights',
     'senna aOR': round(np.exp(model_ow.params['senna_exposed']), 3),
     '95% CI': f"{np.exp(model_ow.conf_int().loc['senna_exposed',0]):.3f}–"
               f"{np.exp(model_ow.conf_int().loc['senna_exposed',1]):.3f}",
     'P': f"{model_ow.pvalues['senna_exposed']:.3g}"},
])
print(summary_iptw.to_string(index=False))
summary_iptw.to_excel('TableS_IPTW_results.xlsx', index=False)
print('\n已保存 TableS_IPTW_results.xlsx')

                    Method  senna aOR      95% CI        P
IPTW (stabilized, trimmed)      1.198 1.138–1.262 6.16e-12
           Overlap weights      1.168 1.082–1.262 7.76e-05

已保存 TableS_IPTW_results.xlsx


In [19]:
# ========== 汇总所有分析 ==========
summary_final = pd.DataFrame([
    {'Analysis': 'Univariable logistic',
     'aOR': 1.265, '95% CI': '1.205–1.328', 'P': '2.21×10⁻²¹'},
    {'Analysis': 'Multivariable logistic (main)',
     'aOR': 1.190, '95% CI': '1.124–1.260', 'P': '2.53×10⁻⁹'},
    {'Analysis': 'Multivariable (excl. drug count)',
     'aOR': 1.203, '95% CI': '1.137–1.272', 'P': '1.33×10⁻¹⁰'},
    {'Analysis': 'PSM (with replacement, caliper=0.02)',
     'aOR': 1.230, '95% CI': '1.173–1.289', 'P': '5.88×10⁻¹⁸'},
    {'Analysis': 'PSM + cluster robust SE',
     'aOR': 1.230, '95% CI': '1.141–1.325', 'P': '5.99×10⁻⁸'},
    {'Analysis': 'IPTW (stabilized, trimmed)',
     'aOR': 1.198, '95% CI': '1.138–1.262', 'P': '6.16×10⁻¹²'},
    {'Analysis': 'Overlap weights',
     'aOR': 1.168, '95% CI': '1.082–1.262', 'P': '7.76×10⁻⁵'},
    {'Analysis': 'Dose–response (High vs Low)',
     'aOR': 1.641, '95% CI': '1.495–1.802', 'P': '2.75×10⁻²⁵'},
])
summary_final.to_excel('Table2_summary_all_analyses.xlsx', index=False)
print(summary_final.to_string(index=False))

                            Analysis   aOR      95% CI          P
                Univariable logistic 1.265 1.205–1.328 2.21×10⁻²¹
       Multivariable logistic (main) 1.190 1.124–1.260  2.53×10⁻⁹
    Multivariable (excl. drug count) 1.203 1.137–1.272 1.33×10⁻¹⁰
PSM (with replacement, caliper=0.02) 1.230 1.173–1.289 5.88×10⁻¹⁸
             PSM + cluster robust SE 1.230 1.141–1.325  5.99×10⁻⁸
          IPTW (stabilized, trimmed) 1.198 1.138–1.262 6.16×10⁻¹²
                     Overlap weights 1.168 1.082–1.262  7.76×10⁻⁵
         Dose–response (High vs Low) 1.641 1.495–1.802 2.75×10⁻²⁵
